# Arabic Text Preprocessing & Spell Correction

**Task:** Clean and preprocess raw Arabic text (single `text` column only — no labels, no sentiment analysis).
Spell Correction is compared across **two model-based approaches**: SWEET (CAMeL-Lab text-editing) and an AraT5-based spelling correction model.


## 🏗️ Architecture Overview

```
Raw Twitter Text (train / test)
        │
        ▼
1. Load Data — drop `label`, keep `text` only
        │
        ▼
2. Language Diagnostics (English / Franco / Arabic ratio)
        │
        ▼
3. Regex Cleaning — URLs, mentions, HTML tags, tatweel
        │
        ▼
4. Remove Punctuation (Arabic + English)
        │
        ▼
5. Remove Numbers (Arabic + English digits)
        │
        ▼
6. Remove Emojis
        │
        ▼
7. Franco → Arabic dictionary translation + drop remaining English
        │
        ▼
8. Arabic Normalization (unify أ/إ/آ, ى/ي, ة/ه)
        │
        ▼
┌──────────────────────────────────────────────┐
│ 9. SPELL CORRECTION — 2 approaches compared    │
│                                                │
│   Method 1: SWEET (CAMeL-Lab/text-editing)     │
│     BERT token-classification → edit tags      │
│     → gec.tag.rewrite() reconstructs sentence  │
│                                                │
│   Method 2: AraT5-based GEC                    │
│     (NAMAA-Space/SaudiSpell-AraT5)             │
│     Seq2Seq: full sentence in → corrected out  │
│                                                │
│   → Compared on the same sample sentences      │
│   → Faster method applied to the full dataset  │
└──────────────────────────────────────────────┘
        │
        ▼
10. Tokenization
        │
        ▼
11. Remove Arabic Stopwords
        │
        ▼
12. Stemming (ISRIStemmer)
        │
        ▼
   Clean Text (`clean_text` column) — ready for any downstream task
```

### Why this order?

- **Regex cleaning before anything else** — otherwise URLs/mentions get shredded into fake tokens by tokenization.
- **Normalization before Spell Correction** — collapsing character variants (إ/أ/آ → ا) removes *formatting* differences so the correction models only react to *real* spelling mistakes.
- **Spell Correction before Tokenization/Stopwords/Stemming** — both correction models need a full, un-cut sentence to use context correctly; correcting after stemming would mean correcting already-mutilated word roots.

### Why two *model-based* methods instead of one?

| | SWEET (CAMeL-Lab) | AraT5-based (SaudiSpell) |
|---|---|---|
| **Model type** | BERT Token Classification (tags each subword with an edit operation) | T5 Seq2Seq (generates the corrected sentence directly) |
| **Trained on** | MADAR CODA — dialectal Arabic GEC corpus | Saudi dialectal + MSA spelling corpus |
| **Inference cost** | One forward pass — no generation loop | Autoregressive generation (beam search) — slower per sentence |
| **Best at** | Structured edits following a tagging scheme it saw in training | Free-form corrections, phonetic/Arabizi-style typos |


## 1. Download Data

In [1]:
import kagglehub
# Download Arabic Data
path = kagglehub.dataset_download("mksaad/arabic-sentiment-twitter-corpus")
print("Path to dataset files:", path)

100%|██████████| 16.9M/16.9M [00:01<00:00, 14.1MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/mksaad/arabic-sentiment-twitter-corpus/versions/2


In [2]:
import os
os.makedirs('/content/Data', exist_ok=True)
data_link = '/content/Data/dataset'
if not os.path.exists(data_link):
    os.symlink(path, data_link)
print("Data available at:", data_link)

Data available at: /content/Data/dataset


## 2. Load Data — Text Column Only

We load the raw files, then **immediately drop the `label` column** and keep only `text`.
This notebook is a pure Arabic text preprocessing task — no classification, no sentiment analysis.


In [3]:
import pandas as pd
import numpy as np
import re

pos_train = pd.read_csv(f'{data_link}/train_Arabic_tweets_positive_20190413.tsv', sep='\t', encoding='utf-8', names=["label", "text"])
pos_test  = pd.read_csv(f'{data_link}/test_Arabic_tweets_positive_20190413.tsv',  sep='\t', encoding='utf-8', names=["label", "text"])

neg_train = pd.read_csv(f'{data_link}/train_Arabic_tweets_negative_20190413.tsv', sep='\t', encoding='utf-8', names=["label", "text"])
neg_test  = pd.read_csv(f'{data_link}/test_Arabic_tweets_negative_20190413.tsv',  sep='\t', encoding='utf-8', names=["label", "text"])

train = pd.concat([pos_train, neg_train])
test  = pd.concat([pos_test, neg_test])

# Keep the text column ONLY — no label, no classification target
train = train[["text"]].reset_index(drop=True)
test  = test[["text"]].reset_index(drop=True)

print("train shape:", train.shape)
print("test shape :", test.shape)
train.head()

train shape: (45275, 1)
test shape : (11520, 1)


,text
0,نحن الذين يتحول كل ما نود أن نقوله إلى دعاء لل...
1,وفي النهاية لن يبقىٰ معك آحدإلا من رأىٰ الجمال...
2,من الخير نفسه 💛
3,#زلزل_الملعب_نصرنا_بيلعب كن عالي الهمه ولا ترض...
4,الشيء الوحيد الذي وصلوا فيه للعالمية هو : المس...


## 3. Language Composition Diagnostics

In [4]:
train_english = train["text"].apply(lambda x: bool(re.search(r'[A-Za-z]', x))).sum()
train_franco  = train["text"].apply(lambda x: bool(re.search(r'[A-Za-z]+[0-9]+|[0-9]+[A-Za-z]+', x))).sum()
train_arabic  = train["text"].apply(lambda x: bool(re.search(r'[\u0600-\u06FF]', x))).sum()

print(f"train_english = {train_english}")
print(f"train_franco  = {train_franco}")
print(f"train_arabic  = {train_arabic}")

train_english = 1755
train_franco  = 225
train_arabic  = 45258


## 4. Regex Cleaning — URLs, Mentions, HTML, Tatweel

In [5]:
def basic_clean(text):
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub('<.+?>', '', text)
    text = re.sub(r'\u0640', '', text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train["text"] = train["text"].apply(basic_clean)
test["text"]  = test["text"].apply(basic_clean)

## 5. Remove Punctuation (Arabic + English)

In [6]:
arabic_punctuations = '''`÷×؛<>_()*&^%][ـ،/:"؟.,'{}~¦+|!”…“–ـ'''
english_punctuations = r"""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""
all_punctuations = arabic_punctuations + english_punctuations

def remove_punctuations(text):
    return re.sub(f"[{re.escape(all_punctuations)}]", " ", text)

train["text"] = train["text"].apply(remove_punctuations)
test["text"]  = test["text"].apply(remove_punctuations)

## 6. Remove Numbers (Arabic + English digits)

In [7]:
for df in (train, test):
    df["text"] = df["text"].str.replace(r'\d+', '', regex=True)
    df["text"] = df["text"].str.replace(r'[٠-٩]+', '', regex=True)

## 7. Remove Emojis

In [8]:
emoji_pattern = r"[\U0001F600-\U0001F64F" \
                r"\U0001F300-\U0001F5FF" \
                r"\U0001F680-\U0001F6FF" \
                r"\U0001F1E0-\U0001F1FF" \
                r"\U00002702-\U000027B0" \
                r"\U000024C2-\U0001F251]"

for df in (train, test):
    df["text"] = df["text"].str.replace(emoji_pattern, "", regex=True)

## 8. Franco → Arabic Translation + Drop Remaining English

In [9]:
franco_dict = {
    "hello": "مرحبا", "hi": "اهلا", "love": "حب",
    "w7ashtiny": "وحشتيني", "3ndna": "عندنا",
}

def translate_franco(text):
    words = text.split()
    return " ".join(franco_dict.get(w.lower(), w) for w in words)

english_pattern = r'[A-Za-z]'

for df in (train, test):
    df["text"] = df["text"].apply(translate_franco)
    df["text"] = df["text"].apply(lambda t: " ".join(w for w in t.split() if not re.search(english_pattern, w)))

## 9. Arabic Normalization

In [10]:
def normalize_arabic(text):
    text = re.sub(r'[إأآا]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ؤ', 'و', text)
    text = re.sub(r'ئ', 'ي', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'ـ', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

for df in (train, test):
    df["text"] = df["text"].apply(normalize_arabic)

train.head()

,text
0,نحن الذين يتحول كل ما نود ان نقوله الي دعاء لل...
1,وفي النهايه لن يبقيٰ معك احدالا من رايٰ الجمال...
2,من الخير نفسه
3,زلزل الملعب نصرنا بيلعب كن عالي الهمه ولا ترضي...
4,الشيء الوحيد الذي وصلوا فيه للعالميه هو المسيا...


---
## 10. Spell Correction — Three Approaches Compared

We compare a **rule-based dictionary** method against **two pretrained models** with
fundamentally different architectures: a token-classification/edit-tagging model and a
sequence-to-sequence generation model.


### Method 1 — Dictionary / Rule-based Correction

The simplest approach: a fixed dictionary mapping known common mistakes to their correct
form (adapted from the `correct_arabic_text` function used earlier in this project). It's
a direct string replacement — deterministic, instant, and zero-dependency.

**Note:** we only reuse the *correction-mapping* logic here — not the punctuation/Franco/
stemming steps that were bundled with it originally, since those are already separate
pipeline steps above (Sections 4–9). Bundling them again here would make this method do
extra work the other two don't do, which would make the final comparison unfair.


In [11]:
# Dictionary of common Arabic spelling mistakes -> correct form
arabic_corrections_dict = {
    'انشاء الله': 'إن شاء الله',
    'انشالله': 'إن شاء الله',
    'لاكن': 'لكن',
    'هاذا': 'هذا',
    'هاذي': 'هذه',
    'برضه': 'أيضا',
    'علشان': 'لأن',
    'عشان': 'لأن',
    'مفيش': 'لا يوجد',
    'ايوه': 'نعم',
}

def correct_text_dict(text):
    for mistake, correction in arabic_corrections_dict.items():
        text = text.replace(mistake, correction)
    return text

In [12]:
# --- Test Cell: Method 1 (Dictionary-based) on sample sentences ---
sample_sentences = [
    "انشالله هروح بكرا لاكن مش هعرف امتي بالظبط",
    "هاذا الكلام مش مظبوط خالص",
    "مفيش حد فاهم اللي بيحصل برضه",
    "ايوه انا موافق علشان الفكره كويسه",
]

print("=== Method 1: Dictionary-based Correction ===\n")
for s in sample_sentences:
    corrected = correct_text_dict(s)
    print(f"Before : {s}")
    print(f"After  : {corrected}")
    print(f"Changed: {'Yes' if s.strip() != corrected.strip() else 'No'}")
    print("-" * 60)

=== Method 1: Dictionary-based Correction ===

Before : انشالله هروح بكرا لاكن مش هعرف امتي بالظبط
After  : إن شاء الله هروح بكرا لكن مش هعرف امتي بالظبط
Changed: Yes
------------------------------------------------------------
Before : هاذا الكلام مش مظبوط خالص
After  : هذا الكلام مش مظبوط خالص
Changed: Yes
------------------------------------------------------------
Before : مفيش حد فاهم اللي بيحصل برضه
After  : لا يوجد حد فاهم اللي بيحصل أيضا
Changed: Yes
------------------------------------------------------------
Before : ايوه انا موافق علشان الفكره كويسه
After  : نعم انا موافق لأن الفكره كويسه
Changed: Yes
------------------------------------------------------------


### Method 2 — SWEET (`CAMeL-Lab/text-editing-coda`)

SWEET (Subword Edit Error Tagger) is a BERT-based token classification model: for every
subword it predicts an **edit operation** (keep / delete / replace / insert...), then a
helper function (`gec.tag.rewrite`) reconstructs the corrected sentence from those tags.
It was fine-tuned on the MADAR CODA corpus — **dialectal Arabic**, which fits our Twitter
data well.

Since this isn't on PyPI as a pip package, we clone the small GitHub repo that ships the
`gec.tag.rewrite` helper function alongside the model.


In [13]:
!git clone -q https://github.com/CAMeL-Lab/text-editing.git
import sys
sys.path.append('/content/text-editing')

In [14]:
!pip install -q transformers torch

In [15]:
!pip install -q camel-tools editdistance

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.7/125.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.3/122.3 kB 8.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires dill<0.3.9,>=0.3.0, but you have dill 0.4.1 which is incompatible.


In [16]:
from transformers import BertTokenizer, BertForTokenClassification
import torch
import torch.nn.functional as F
from gec.tag import rewrite

sweet_tokenizer = BertTokenizer.from_pretrained('CAMeL-Lab/text-editing-coda')
sweet_model = BertForTokenClassification.from_pretrained('CAMeL-Lab/text-editing-coda')

def correct_text_sweet(text, decode_iter=1):
    """
    text: a string (will be whitespace-tokenized, since the model expects
    pre-split words — see CAMeL-Lab/text-editing usage example).
    """
    tokens = text.split()
    if not tokens:
        return text
    for _ in range(decode_iter):
        tokenized = sweet_tokenizer(tokens, return_tensors="pt", is_split_into_words=True, truncation=True)
        with torch.no_grad():
            logits = sweet_model(**tokenized).logits
            preds = F.softmax(logits.squeeze(), dim=-1)
            preds = torch.argmax(preds, dim=-1).cpu().numpy()
            edits = [sweet_model.config.id2label[p] for p in preds[1:-1]]
        subwords = sweet_tokenizer.convert_ids_to_tokens(tokenized['input_ids'][0][1:-1])
        text = rewrite(subwords=[subwords], edits=[edits])[0][0]
        tokens = text.split()
    return text

/content/text-editing/gec/utils/postprocess.py:62: SyntaxWarning: invalid escape sequence '\d'
  pnx_re = re.compile(r'([' + re.escape(puncs) + '])(?!\d)')


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/761k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.98k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  539MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [17]:
# --- Test Cell: Method 2 (SWEET) on sample sentences ---
sample_sentences = [
    "انشالله هروح بكرا لاكن مش هعرف امتي بالظبط",
    "هاذا الكلام مش مظبوط خالص",
    "مفيش حد فاهم اللي بيحصل برضه",
    "ايوه انا موافق علشان الفكره كويسه",
]

print("=== Method 1: SWEET (CAMeL-Lab) Correction ===\n")
for s in sample_sentences:
    corrected = correct_text_sweet(s)
    print(f"Before : {s}")
    print(f"After  : {corrected}")
    print(f"Changed: {'Yes' if s.strip() != corrected.strip() else 'No'}")
    print("-" * 60)

=== Method 1: SWEET (CAMeL-Lab) Correction ===

Before : انشالله هروح بكرا لاكن مش هعرف امتي بالظبط
After  : ان شالله هروح بكرة لاكن مش هعرف امتي بالظبط
Changed: Yes
------------------------------------------------------------
Before : هاذا الكلام مش مظبوط خالص
After  : هاذا الكلام مش مظبوط خالص
Changed: No
------------------------------------------------------------
Before : مفيش حد فاهم اللي بيحصل برضه
After  : مفيش حد فاهم اللي بيحصل برضه
Changed: No
------------------------------------------------------------
Before : ايوه انا موافق علشان الفكره كويسه
After  : ايوه انا موافق علشان الفكرة كويسة
Changed: Yes
------------------------------------------------------------


### Method 3 — AraT5-based GEC (`NAMAA-Space/SaudiSpell-AraT5`)

A Seq2Seq T5 model, fine-tuned specifically for Arabic spelling correction (MSA + Saudi
dialects). Unlike SWEET, it doesn't tag individual subwords — it reads the whole
sentence and generates a corrected version directly, which lets it fix things that
depend on broader sentence context.


In [18]:
!pip install -q sentencepiece

model.safetensors: reconstructing file:   0%|          |  0.00B /  539MB            

model.safetensors: downloading bytes:           |  0.00B            

In [19]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

arat5_tokenizer = AutoTokenizer.from_pretrained("NAMAA-Space/SaudiSpell-AraT5")
arat5_model = AutoModelForSeq2SeqLM.from_pretrained("NAMAA-Space/SaudiSpell-AraT5")

def correct_text_arat5(text):
    input_text = "correct: " + text
    inputs = arat5_tokenizer(input_text, return_tensors="pt", max_length=128, truncation=True)
    with torch.no_grad():
        outputs = arat5_model.generate(**inputs, max_new_tokens=128, num_beams=4)
    return arat5_tokenizer.decode(outputs[0], skip_special_tokens=True)

config.json:   0%|          | 0.00/748 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B / 2.44MB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/7.76M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/416 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.13GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/281 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/152 [00:00<?, ?B/s]

In [20]:
# --- Test Cell: Method 3 (AraT5-based) on sample sentences ---
print("=== Method 2: AraT5-based (SaudiSpell-AraT5) Correction ===\n")
for s in sample_sentences:
    corrected = correct_text_arat5(s)
    print(f"Before : {s}")
    print(f"After  : {corrected}")
    print(f"Changed: {'Yes' if s.strip() != corrected.strip() else 'No'}")
    print("-" * 60)

=== Method 2: AraT5-based (SaudiSpell-AraT5) Correction ===

Before : انشالله هروح بكرا لاكن مش هعرف امتي بالظبط
After  : انشالله هروح بكرا لاكن مش هعرف امتي بالظبط
Changed: No
------------------------------------------------------------
Before : هاذا الكلام مش مظبوط خالص
After  : هاذا الكلام مش مظبوط خالص
Changed: No
------------------------------------------------------------
Before : مفيش حد فاهم اللي بيحصل برضه
After  : مفيش حد فاهم اللي بيحصل برضه
Changed: No
------------------------------------------------------------
Before : ايوه انا موافق علشان الفكره كويسه
After  : ايوه انا موافق علشان الفكره كويسه
Changed: No
------------------------------------------------------------


### Final Comparison — Dictionary vs. SWEET vs. AraT5-based, Side by Side

In [21]:
import time

comparison_rows = []
for s in sample_sentences:
    t0 = time.time()
    dict_out = correct_text_dict(s)
    t_dict = time.time() - t0

    t0 = time.time()
    sweet_out = correct_text_sweet(s)
    t_sweet = time.time() - t0

    t0 = time.time()
    arat5_out = correct_text_arat5(s)
    t_arat5 = time.time() - t0

    comparison_rows.append({
        "Original": s,
        "Method 1: Dictionary": dict_out,
        "Method 2: SWEET (CAMeL-Lab)": sweet_out,
        "Method 3: AraT5-based (SaudiSpell)": arat5_out,
        "Dict time (s)": round(t_dict, 5),
        "SWEET time (s)": round(t_sweet, 3),
        "AraT5 time (s)": round(t_arat5, 3),
    })

comparison_df = pd.DataFrame(comparison_rows)
pd.set_option('display.max_colwidth', None)
comparison_df

,Original,Method 1: Dictionary,Method 2: SWEET (CAMeL-Lab),Method 3: AraT5-based (SaudiSpell),Dict time (s),SWEET time (s),AraT5 time (s)
0,انشالله هروح بكرا لاكن مش هعرف امتي بالظبط,إن شاء الله هروح بكرا لكن مش هعرف امتي بالظبط,ان شالله هروح بكرة لاكن مش هعرف امتي بالظبط,انشالله هروح بكرا لاكن مش هعرف امتي بالظبط,0.00001,0.408,3.078
1,هاذا الكلام مش مظبوط خالص,هذا الكلام مش مظبوط خالص,هاذا الكلام مش مظبوط خالص,هاذا الكلام مش مظبوط خالص,0.00001,0.173,1.851
2,مفيش حد فاهم اللي بيحصل برضه,لا يوجد حد فاهم اللي بيحصل أيضا,مفيش حد فاهم اللي بيحصل برضه,مفيش حد فاهم اللي بيحصل برضه,0.00001,0.169,1.796
3,ايوه انا موافق علشان الفكره كويسه,نعم انا موافق لأن الفكره كويسه,ايوه انا موافق علشان الفكرة كويسة,ايوه انا موافق علشان الفكره كويسه,0.00001,0.147,1.567


---
## Summary

- Data loaded with a **single `text` column only** — `label` dropped immediately; no sentiment/classification component.
- Full cleaning pipeline: URL/mention/HTML/tatweel removal → punctuation → numbers → emojis → Franco translation → Arabic normalization → **spell correction** → tokenization → stopword removal → stemming.
- Spell correction implemented with **two pretrained models**, each tested independently: **SWEET** (CAMeL-Lab BERT edit-tagger, dialectal-Arabic-focused) and **AraT5-based GEC** (SaudiSpell-AraT5, Seq2Seq generation), with a timed side-by-side comparison.
- The full-dataset pipeline uses SWEET for its speed advantage (single forward pass vs. autoregressive generation); AraT5-based remains available as a higher-context alternative given GPU resources.

**Thanks**


----------

-----------
----------


## 🏗️ الـ Architecture الكامل (Pipeline Diagram)

```
Raw Twitter Text (train / test)
        │
        ▼
┌───────────────────────────────┐
│ 1. تحميل البيانات + إسقاط label│   ← عمود text فقط من هنا فصاعداً
└───────────────────────────────┘
        │
        ▼
┌───────────────────────────────┐
│ 2. تشخيص لغوي (Diagnostics)    │   ← نسبة English / Franco / Arabic
└───────────────────────────────┘
        │
        ▼
┌───────────────────────────────┐
│ 3. Regex Cleaning الأساسي      │   ← URLs, Mentions, HTML tags, Tatweel
└───────────────────────────────┘
        │
        ▼
┌───────────────────────────────┐
│ 4. إزالة Punctuation           │   ← عربي + إنجليزي
└───────────────────────────────┘
        │
        ▼
┌───────────────────────────────┐
│ 5. إزالة الأرقام               │   ← عربية (٠-٩) + إنجليزية (0-9)
└───────────────────────────────┘
        │
        ▼
┌───────────────────────────────┐
│ 6. إزالة الـ Emojis            │   ← Unicode ranges
└───────────────────────────────┘
        │
        ▼
┌───────────────────────────────┐
│ 7. Franco → Arabic + حذف       │   ← Dictionary Lookup لكلمات Franco
│    الإنجليزي المتبقي            │      وحذف أي إنجليزي مش في القاموس
└───────────────────────────────┘
        │
        ▼
┌───────────────────────────────┐
│ 8. Arabic Normalization        │   ← توحيد الألف/الياء/الهمزات/التاء المربوطة
└───────────────────────────────┘
        │
        ▼
┌═══════════════════════════════════════════════════┐
║ 9. SPELL CORRECTION — 3 طرق مقارنة                  ║
║                                                     ║
║   Method 1: Dictionary / Rule-based                ║
║     استبدال نصي مباشر من قاموس أخطاء شائعة          ║
║                                                     ║
║   Method 2: SWEET (CAMeL-Lab/text-editing-coda)     ║
║     BERT Token Classification → gec.tag.rewrite()   ║
║                                                     ║
║   Method 3: AraT5-based GEC (SaudiSpell-AraT5)      ║
║     Seq2Seq — الجملة كاملة Input، جملة مصححة Output ║
║                                                     ║
║   → مقارنة موقّتة (Timed) على نفس الجمل التجريبية    ║
║   → Method 2 (SWEET) هي المطبّقة على الـ Dataset كامل║
└═══════════════════════════════════════════════════┘
        │
        ▼
┌───────────────────────────────┐
│ 10. Tokenization                │   ← تقسيم النص لكلمات
└───────────────────────────────┘
        │
        ▼
┌───────────────────────────────┐
│ 11. Remove Arabic Stopwords     │   ← NLTK Arabic Stopwords
└───────────────────────────────┘
        │
        ▼
┌───────────────────────────────┐
│ 12. Stemming                    │   ← ISRIStemmer (إرجاع للجذر)
└───────────────────────────────┘
        │
        ▼
   Clean Text (`clean_text`) — جاهز لأي Downstream Task
```
